# Phase 3 — Statistical Comparison

**Goal:** turn the descriptive findings from Phase 2 into rigorous claims that can survive an audience question. For every metric and every veracity comparison, we want to know:

1. **Is there a difference at all?** Mann-Whitney U with Cliff's delta effect size.
2. **Is the difference consistent across events, or driven by one event?** Per-event consistency scoring.
3. **What's the pooled effect after controlling for event differences?** van Elteren stratified test.
4. **How precise are the per-event medians?** Bootstrap 95% CIs.

**Why this much rigor matters for the presentation.** Phase 2 showed that pooled comparisons can be misleading on PHEME (non-rumours look biggest in pooled data, but the ranking varies by event). With ~6,400 threads, p-values on any comparison will be tiny — meaning we can't use p-values alone to decide what's interesting. We need *effect sizes* (how big is the difference?) and *consistency scores* (does the pattern hold across events, or is one event driving everything?).

**The consistency score is the killer slide format.** A finding like *"unverified rumours generate replies faster than non-rumours in 5 of 5 eligible events, all p<0.01, median Cliff's delta = 0.18"* is much harder to argue with than *"p < 0.001"* in a single pooled test.

**Outputs (saved to `data/processed/`):**
- `stats_per_event.parquet` — every (event, metric, pair) test result
- `stats_pooled.parquet` — van Elteren stratified results
- `stats_consistency.parquet` — consistency scores by metric × pair
- `medians_with_ci.parquet` — per-event medians with bootstrap CIs (for Phase 4)

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve() / 'src'))
from stats_comparison import (
    per_event_pairwise, van_elteren_test,
    consistency_score, per_event_medians_with_ci,
)

DATA_DIR = Path('../data/processed')
sns.set_style('whitegrid')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 200)

VERACITY_ORDER = ['nonrumour', 'true', 'unverified', 'false']
VERACITY_COLORS = {
    'nonrumour':  '#4C72B0',
    'true':       '#55A868',
    'unverified': '#DD8452',
    'false':      '#C44E52',
}

In [ ]:
# Load Phase 2 output
def _load(name):
    pq, pkl = DATA_DIR / f'{name}.parquet', DATA_DIR / f'{name}.pkl'
    if pq.exists():
        return pd.read_parquet(pq)
    return pd.read_pickle(pkl)

tm = _load('threads_with_metrics')
print(f'Loaded {len(tm):,} threads with {len(tm.columns)} columns')

METRICS = [
    'cascade_size', 'max_depth', 'max_breadth', 'unique_users',
    'time_to_first_reply_min', 'time_to_half_cascade_min',
    'reply_velocity_first_hour',
    'structural_virality', 'wiener_index', 'broadcast_ratio',
    'branching_factor_mean',
]

# The pairs we want to test. We exclude (true, unverified) because it's
# of less narrative interest and because keeping the matrix manageable
# matters more than completeness.
VERACITY_PAIRS = [
    ('unverified', 'nonrumour'),
    ('false', 'nonrumour'),
    ('true', 'nonrumour'),
    ('false', 'true'),
    ('false', 'unverified'),
]

## 2. Per-event pairwise tests for all metrics

For each (event, metric, pair), we run Mann-Whitney U + Cliff's delta. Eligibility threshold: each group must have ≥30 observations. Below that, effect size estimates are noisy and Mann-Whitney's normal approximation breaks down.

In [ ]:
all_results = []
for metric in METRICS:
    res = per_event_pairwise(tm, metric, VERACITY_PAIRS, min_n_per_group=30)
    all_results.append(res)
stats_per_event = pd.concat(all_results, ignore_index=True)

# Quick summary of how many comparisons were eligible
elig_summary = stats_per_event.groupby('metric')['eligible'].agg(
    n_total='count', n_eligible='sum'
)
elig_summary['pct_eligible'] = (elig_summary['n_eligible'] / elig_summary['n_total'] * 100).round(0)
print('How many (event × pair) comparisons were eligible per metric:')
print(elig_summary)

**Reading this table:** higher `pct_eligible` means fewer events were dropped for low sample sizes on that metric. Speed metrics (`time_to_first_reply_min`, etc.) are computed only on threads with timestamps, so they may have lower eligibility than reach metrics.

## 3. Consistency scoring — the headline finding format

For each (metric, pair), how many of the eligible events showed a significant difference in each direction? This is the table to mine for the presentation.

In [ ]:
consistency = consistency_score(stats_per_event, alpha=0.05)

# Sort by absolute consistency (how clear the pattern is)
consistency['max_consistency'] = consistency[['consistency_a_gt_b', 'consistency_b_gt_a']].max(axis=1)
consistency_sorted = consistency.sort_values('max_consistency', ascending=False)

# Show only rows where at least 3 events were eligible (otherwise the
# percentage is meaningless)
presentable = consistency_sorted[consistency_sorted['n_eligible'] >= 3].copy()
presentable['summary'] = presentable.apply(
    lambda r: f"{r['group_a']} > {r['group_b']} in {r['n_a_greater']}/{r['n_eligible']} events"
              if r['n_a_greater'] >= r['n_b_greater']
              else f"{r['group_b']} > {r['group_a']} in {r['n_b_greater']}/{r['n_eligible']} events",
    axis=1,
)

print('=== Strongest consistency findings (sorted) ===')
print(presentable[['metric', 'summary', 'median_delta', 'n_eligible']].to_string(index=False))

**How to read this table for the presentation.** Each row is a candidate finding. Pick the rows where:

1. The summary fraction is high (e.g., 4/5 or 5/5),
2. `n_eligible` is at least 3 (less than that, you can't claim a pattern),
3. `median_delta` is at least 0.15 in magnitude (Romano et al. 'small' threshold — anything below is statistical artifact territory).

The top 3-5 rows of this table are likely the central findings for the presentation.

## 4. Pooled stratified tests (van Elteren)

For each (metric, pair), the pooled stratified test gives us a single combined p-value and weighted Cliff's delta across all eligible events. Useful for sentences like *"after controlling for event-level differences, unverified rumours have N% higher cascade size than non-rumours (combined p<0.001, weighted delta=0.18)"*.

In [ ]:
pooled_rows = []
for metric in METRICS:
    for a, b in VERACITY_PAIRS:
        result = van_elteren_test(tm, metric, a, b, stratum_col='event', min_n_per_group=30)
        pooled_rows.append({
            'metric': metric, 'group_a': a, 'group_b': b,
            **result,
        })
stats_pooled = pd.DataFrame(pooled_rows)

# Show the strongest pooled effects
stats_pooled['abs_delta'] = stats_pooled['weighted_delta'].abs()
stats_pooled_sorted = stats_pooled.sort_values('abs_delta', ascending=False)

# Filter to results where at least 3 events were used in pooling
presentable_pooled = stats_pooled_sorted[stats_pooled_sorted['n_strata'] >= 3]

print('=== Strongest pooled effects (n_strata ≥ 3, sorted by |delta|) ===')
print(presentable_pooled[[
    'metric', 'group_a', 'group_b', 'weighted_delta', 'combined_p', 'n_strata'
]].head(20).to_string(index=False))

**Interpretation tips:**

- `weighted_delta > 0` means group_a tends to have higher values than group_b (after stratification).
- |delta| < 0.15 is negligible regardless of p-value. With this much data, p will always be tiny.
- Combine pooled findings with the consistency table: if pooled says "unverified > nonrumour, delta=0.2" AND consistency says "4/5 events," you have a robust finding.

## 5. Per-event medians with bootstrap CIs (for Phase 4 figures)

For each metric × event × veracity combination, compute the median and a 95% bootstrap CI. We'll plot these as a per-event panel figure in Phase 4.

In [ ]:
medians_rows = []
for metric in METRICS:
    res = per_event_medians_with_ci(tm, metric, VERACITY_ORDER, n_boot=1000)
    res['metric'] = metric
    medians_rows.append(res)
medians_with_ci = pd.concat(medians_rows, ignore_index=True)

print(f'Computed medians + CIs for {len(medians_with_ci)} (metric, event, veracity) combinations')
print()
print('Sample (cascade_size, top events):')
print(medians_with_ci[
    (medians_with_ci['metric'] == 'cascade_size') &
    (medians_with_ci['n'] >= 30)
].head(20).to_string(index=False))

## 6. Headline figure — consistency at a glance

A single figure that summarizes the core findings: for each metric, what fraction of eligible events showed each direction? This is a strong candidate for the methodology slide of the presentation.

In [ ]:
# Pick the most narratively important pair: unverified vs nonrumour.
# (Customize this loop to highlight whichever pair the data favors.)
focus_pair = ('unverified', 'nonrumour')
focus = consistency[
    (consistency['group_a'] == focus_pair[0]) &
    (consistency['group_b'] == focus_pair[1]) &
    (consistency['n_eligible'] >= 3)
].copy()
focus = focus.sort_values('consistency_a_gt_b', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(focus))
ax.barh(y, focus['n_a_greater'], color=VERACITY_COLORS[focus_pair[0]],
        label=f'{focus_pair[0]} > {focus_pair[1]}')
ax.barh(y, focus['n_b_greater'], left=focus['n_a_greater'],
        color=VERACITY_COLORS[focus_pair[1]],
        label=f'{focus_pair[1]} > {focus_pair[0]}')
ax.barh(y, focus['n_no_diff'],
        left=focus['n_a_greater'] + focus['n_b_greater'],
        color='#cccccc', label='no significant difference')
ax.set_yticks(y)
ax.set_yticklabels(focus['metric'])
ax.set_xlabel('Number of eligible events (out of N)')
ax.set_title(f'Consistency of {focus_pair[0]} vs {focus_pair[1]} across events\n(per-event Mann-Whitney p<0.05)')
ax.legend(loc='lower right', frameon=True)
plt.tight_layout()
Path('../figures').mkdir(exist_ok=True)
plt.savefig('../figures/phase3_consistency_unverified_vs_nonrumour.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Save outputs

In [ ]:
try:
    stats_per_event.to_parquet(DATA_DIR / 'stats_per_event.parquet', index=False)
    stats_pooled.to_parquet(DATA_DIR / 'stats_pooled.parquet', index=False)
    consistency.to_parquet(DATA_DIR / 'stats_consistency.parquet', index=False)
    medians_with_ci.to_parquet(DATA_DIR / 'medians_with_ci.parquet', index=False)
    ext = 'parquet'
except ImportError:
    stats_per_event.to_pickle(DATA_DIR / 'stats_per_event.pkl')
    stats_pooled.to_pickle(DATA_DIR / 'stats_pooled.pkl')
    consistency.to_pickle(DATA_DIR / 'stats_consistency.pkl')
    medians_with_ci.to_pickle(DATA_DIR / 'medians_with_ci.pkl')
    ext = 'pkl'

print(f'Saved 4 files as .{ext} to', DATA_DIR.resolve())

---

## What to do next

Look at section 3 (consistency table). The **top 3-5 rows** are your candidate findings for the presentation. Pick the strongest 1-2 to anchor the story.

A good central finding has all of:
- High consistency (4/5 or 5/5 events)
- |median_delta| ≥ 0.15
- A natural narrative interpretation ("X spreads more virally than Y" beats "X has higher metric Z than Y")

Phase 4 builds the six presentation figures around whatever survives this filter. Send me your top 3 findings from the consistency table and I'll design the figures around them.
